In [ ]:
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.85'
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
#os.makedirs('/content/drive/MyDrive/chess_vae_checkpoints', exist_ok=True)
print("Drive mounted. Checkpoint directory ready.")

Mounted at /content/drive
Drive mounted. Checkpoint directory ready.


In [ ]:
%cd /content
!mkdir -p /content/searchless_chess_ml_opt

/content


In [ ]:
!git clone https://github.com/jameswu05/searchless_chess_ml_opt.git
%cd searchless_chess_ml_opt
!pip install -r requirements.txt -q
!pip install --upgrade "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html -q
print("Done.")

fatal: destination path 'searchless_chess_ml_opt' already exists and is not an empty directory.
/content/searchless_chess_ml_opt
Done.


In [ ]:
import jax
print(jax.devices())
!nvidia-smi | head -20

[CudaDevice(id=0)]
Mon Apr  6 03:36:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             52W /  400W |     436MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------

In [ ]:
import sys
os.environ['PYTHONPATH'] = '/content'
sys.path.insert(0, '/content')

!rm -f /content/searchless_chess
!ln -sf /content/searchless_chess_ml_opt /content/searchless_chess

# Verify
!ls /content/searchless_chess/src/transformer.py
print("PYTHONPATH and symlink OK.")

/content/searchless_chess/src/transformer.py
PYTHONPATH and symlink OK.


In [ ]:
!touch /content/searchless_chess_ml_opt/__init__.py
!touch /content/searchless_chess_ml_opt/src/__init__.py
!touch /content/searchless_chess_ml_opt/src/engines/__init__.py
print("__init__.py files created.")

__init__.py files created.


In [ ]:
%cd /content/searchless_chess_ml_opt

os.makedirs('data/train', exist_ok=True)
os.makedirs('data/test', exist_ok=True)

print("Downloading puzzles...")
!wget -q https://storage.googleapis.com/searchless_chess/data/puzzles.csv -P data/

print("Downloading test set (141MB)...")
!wget -q https://storage.googleapis.com/searchless_chess/data/test/action_value_data.bag -P data/test/

print("Downloading training shard (1.2GB)...")
!wget -q "https://storage.googleapis.com/searchless_chess/data/train/action_value-00000-of-02148_data.bag" -P data/train/
!mv data/train/action_value-00000-of-02148_data.bag data/train/action_value_data.bag

print("Done.")
!ls data/train/
!ls data/test/

/content/searchless_chess_ml_opt
Done.
action_value_data.bag
action_value_data.bag	 action_value_data.bag.2  action_value_data.bag.4
action_value_data.bag.1  action_value_data.bag.3  action_value_data.bag.5


In [ ]:
# Read current training.py and patch the checkpoint directory
with open('/content/searchless_chess_ml_opt/src/training.py', 'r') as f:
    content = f.read()

# Replace checkpoint directory to point to Google Drive
old = "checkpoint_dir = os.path.join(\n        os.getcwd(),\n        f'../checkpoints/local/{train_config.data.policy}',\n    )"
new = "checkpoint_dir = f'/content/drive/MyDrive/chess_vae_checkpoints/{train_config.data.policy}'"

content = content.replace(old, new)

with open('/content/searchless_chess_ml_opt/src/training.py', 'w') as f:
    f.write(content)

# Verify the change
!grep -n "checkpoint_dir" /content/searchless_chess_ml_opt/src/training.py

106:    checkpoint_dir = f'/content/drive/MyDrive/chess_vae_checkpoints/{train_config.data.policy}'
111:        checkpoint_dir=checkpoint_dir,
260:    checkpoint_dir = f'/content/drive/MyDrive/chess_vae_checkpoints/{train_config.data.policy}'
265:        checkpoint_dir=checkpoint_dir,


In [ ]:
%cd /content/searchless_chess_ml_opt/src
!PYTHONPATH=/content python3 test_transformer.py

/content/searchless_chess_ml_opt/src
Running transformer unit tests...

E0406 03:37:06.956308  117147 xtile_compiler.cc:399] Fusion: gemm_fusion_dot = f32[316,64]{1,0} fusion(bitcast.6, b.1), kind=kCustom, calls=gemm_fusion_dot_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0406 03:37:06.956499  117147 xtile_compiler.cc:401] Computation: gemm_fusion_dot_computation.clone {
  parameter_0 = f32[316,256]{1,0} parameter(0)
  parameter_1 = f32[256,64]{1,0} parameter(1)
  ROOT dot.1 = f32[316,64]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["64"]}
}
E0406 0

In [ ]:
!grep "sharding.replicate" /content/searchless_chess_ml_opt/src/training_utils.py
!grep -R "ckpt_frequency=" /content/searchless_chess_ml_opt
!find /content/searchless_chess_ml_opt -name "train.py"

          array.shape, sharding.replicate(), lambda _: array
            sharding = sharding.replicate() if sharding is not None else None,
/content/searchless_chess_ml_opt/src/searchless_chess_ml_opt/src/training.py:        ckpt_frequency=train_config.ckpt_frequency,
/content/searchless_chess_ml_opt/src/searchless_chess_ml_opt/src/training.py:        ckpt_frequency=train_config.ckpt_frequency,
/content/searchless_chess_ml_opt/src/searchless_chess_ml_opt/src/training_utils.py:        f' save_frequency={save_frequency} and ckpt_frequency={ckpt_frequency}.'
/content/searchless_chess_ml_opt/src/searchless_chess_ml_opt/src/train.py:      ckpt_frequency=10_000,
grep: /content/searchless_chess_ml_opt/src/__pycache__/training_utils.cpython-312.pyc: binary file matches
/content/searchless_chess_ml_opt/src/training.py:        ckpt_frequency=train_config.ckpt_frequency,
/content/searchless_chess_ml_opt/src/training.py:        ckpt_frequency=train_config.ckpt_frequency,
/content/searchless_chess_

In [ ]:
!mv "/content/drive/MyDrive/chess_vae_checkpoints/action_value (1)" \
     "/content/drive/MyDrive/chess_vae_checkpoints/action_value"
!ls -la "/content/drive/MyDrive/chess_vae_checkpoints/action_value"
!ls -la "/content/drive/MyDrive/chess_vae_checkpoints/action_value (1)"

mv: cannot stat '/content/drive/MyDrive/chess_vae_checkpoints/action_value (1)': No such file or directory
total 56
drwx------ 2 root root 4096 Apr  5 21:22 0
drwx------ 2 root root 4096 Apr  5 21:22 100000
drwx------ 2 root root 4096 Apr  5 21:22 150000
drwx------ 2 root root 4096 Apr  5 21:22 200000
drwx------ 2 root root 4096 Apr  5 21:22 250000
drwx------ 2 root root 4096 Apr  5 21:22 300000
drwx------ 2 root root 4096 Apr  5 21:22 350000
drwx------ 2 root root 4096 Apr  5 21:22 400000
drwx------ 2 root root 4096 Apr  5 21:22 450000
drwx------ 2 root root 4096 Apr  5 21:22 50000
drwx------ 2 root root 4096 Apr  5 21:22 500000
drwx------ 2 root root 4096 Apr  5 23:35 550000
drwx------ 2 root root 4096 Apr  6 01:51 600000
drwx------ 2 root root 4096 Apr  6 03:13 630000
ls: cannot access '/content/drive/MyDrive/chess_vae_checkpoints/action_value (1)': No such file or directory


In [ ]:
'''

file_path = "/content/searchless_chess/src/training_utils.py"

with open(file_path, "r") as f:
    content = f.read()

content = content.replace(
    "sharding=sharding.replicate(),",
    "sharding = sharding.replicate() if sharding is not None else None,"
)

with open(file_path, "w") as f:
    f.write(content)

print("Patched training_utils.py")

'''

'\n\nfile_path = "/content/searchless_chess/src/training_utils.py"\n\nwith open(file_path, "r") as f:\n    content = f.read()\n\ncontent = content.replace(\n    "sharding=sharding.replicate(),",\n    "sharding = sharding.replicate() if sharding is not None else None,"\n)\n\nwith open(file_path, "w") as f:\n    f.write(content)\n\nprint("Patched training_utils.py")\n\n'

In [ ]:
%cd /content/searchless_chess_ml_opt/src
!PYTHONPATH=/content python3 train.py --policy=action_value

/content/searchless_chess_ml_opt/src
I0406 03:37:31.360837 135582212854400 xla_bridge.py:822] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0406 03:37:31.661571 135582212854400 training.py:198] [Process 0]: Using 1 processes with 1 local devices each.
I0406 03:37:31.663998 135582212854400 training.py:211] Initializing the predictor parameters.
E0406 03:37:34.758178  117810 xtile_compiler.cc:399] Fusion: gemm_fusion_dot = f32[79,256]{1,0} fusion(bitcast.6, b.1), kind=kCustom, calls=gemm_fusion_dot_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEV